# Peel-CQR v3 — extended benchmark

Additions over v2, aimed at a machine-learning / AI-applications venue:

1. **Uncertainty baselines from the ML literature:** weighted conformal prediction under covariate shift (density ratio estimated by a classifier), quantile regression forests, CV+ (cross-conformal), an MLP ensemble, and the Gaussian process.
2. **Learner-agnostic Peel-CQR:** run with gradient-boosted quantiles *and* with quantile regression forests.
3. **Simulation study** on a synthetic chart with known truth and a tunable extrapolation difficulty κ.
4. **Interval score** (Winkler), **conditional coverage** by physical region, **Holm-corrected** McNemar tests.
5. **Ablations:** distance metric (Manhattan / Euclidean / Chebyshev), monotone constraint on the calibration curve, number of layers K; **computation time** per method.
6. **Application:** ranking unmeasured nuclei for future measurement by calibrated uncertainty.
7. **Speed:** GPU (XGBoost, PyTorch) when available and parallel workers for folds, ablations and simulation replications.

Everything is saved to `nuclear_mass_results_v3.xlsx`. AME2012 and AME2020 are loaded from the repository; AME2003 and AME2016 add folds when placed in `ame_data/`.

In [ ]:
import os, glob, shutil, time, warnings, urllib.request, subprocess, sys
for pkg in ["openpyxl"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
from sklearn.model_selection import train_test_split, KFold

warnings.filterwarnings("ignore")
SEED = 42
ALPHA = 0.10
K_PEEL = 3
MIN_Z, MIN_N = 8, 8
DIST_METRIC = "manhattan"     # changed only inside the ablation
RUN_GP = True
RUN_SIMULATION = True
N_SIM_REPS = 10
SIM_KAPPAS = [0.0, 0.5, 1.0]
DATA_DIR = "ame_data"
os.makedirs(DATA_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## Compute backend: GPU and parallel workers

Settings in the next cell control speed; they do **not** change the method.

| Setting | Default | Effect |
|---|---|---|
| `USE_GPU` | `"auto"` | use NVIDIA GPU(s) when present (Colab, Kaggle, a local CUDA card) |
| `N_JOBS` | `-1` | parallel worker processes (`-1` = all CPU cores) |
| `GPU_WORKERS_PER_GPU` | `3` | how many workers share one GPU (each holds a CUDA context, ~0.3–0.5 GB) |

What runs where:

* **Gradient-boosted quantiles** → XGBoost (`reg:quantileerror`, all quantile levels in one model) on the GPU; scikit-learn `HistGradientBoostingRegressor` on CPU.
* **Gaussian process** → exact GP in PyTorch on the GPU (Adam on the marginal likelihood, Cholesky solves); scikit-learn on CPU.
* **MLP ensemble** → PyTorch on the GPU; scikit-learn on CPU.
* **Quantile regression forest** weights → PyTorch on the GPU.
* **Independent runs** (folds, subsamples, ablations, simulation replications) → spread over worker processes with `joblib`; with several GPUs, workers alternate between them (Kaggle's 2×T4 are both used).

Any GPU step that fails falls back to the CPU implementation and prints one warning. The GPU and CPU learners are different implementations, so numbers can differ slightly between backends (not between parallel and sequential runs on the same backend). The data set is small (~2,500 nuclei), so the largest gains come from parallel workers and from the GP; a single gradient-boosting fit is about as fast on CPU as on GPU.

In [ ]:
# ---------------------------------------------------------------- compute backend
import os, time, warnings, functools
import numpy as np
from joblib import Parallel, delayed

USE_GPU = "auto"            # "auto" | True | False
N_JOBS = -1                 # -1 = all cores
GPU_WORKERS_PER_GPU = 3

try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
try:
    import xgboost as xgb
    HAS_XGB = int(xgb.__version__.split(".")[0]) >= 2      # multi-quantile objective needs XGBoost >= 2.0
except ImportError:
    HAS_XGB = False


def _count_gpus():
    if HAS_TORCH:
        try:
            return torch.cuda.device_count() if torch.cuda.is_available() else 0
        except Exception:
            return 0
    try:
        import subprocess
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10).stdout
        return out.count("GPU ")
    except Exception:
        return 0


N_GPUS = _count_gpus() if USE_GPU in ("auto", True) else 0
GPU_ON = N_GPUS > 0 and (HAS_TORCH or HAS_XGB)
if USE_GPU is True and not GPU_ON:
    print("USE_GPU=True but no usable GPU (or neither torch nor xgboost>=2 installed): using CPU")
N_CPU = os.cpu_count() or 1
N_WORKERS = N_CPU if N_JOBS in (-1, None) else max(1, int(N_JOBS))
DEFAULT_DEVICE = "cuda:0" if GPU_ON else "cpu"
_WARNED = set()


def _warn_once(key, msg):
    if key not in _WARNED:
        _WARNED.add(key); print(f"[backend] {msg} -> falling back to CPU")


def device_for(i):
    return f"cuda:{i % N_GPUS}" if GPU_ON else "cpu"


def pmap(fn, tasks, desc="", n_jobs=None):
    # run fn(**task, device=...) for every task dict; results in task order
    n = len(tasks)
    nj = n_jobs or (min(N_WORKERS, GPU_WORKERS_PER_GPU * N_GPUS) if GPU_ON else N_WORKERS)
    nj = max(1, min(nj, n))
    t0 = time.time()
    if nj == 1:
        res = [fn(**t, device=device_for(i)) for i, t in enumerate(tasks)]
    else:
        res = Parallel(n_jobs=nj, backend="loky")(delayed(fn)(**t, device=device_for(i)) for i, t in enumerate(tasks))
    if desc:
        print(f"{desc}: {n} tasks on {nj} worker(s) in {time.time() - t0:.0f}s")
    return res


# ---- quantile gradient boosting: XGBoost on GPU, scikit-learn on CPU
from sklearn.ensemble import HistGradientBoostingRegressor
GB_KW = dict(max_iter=600, learning_rate=0.04, max_leaf_nodes=15, min_samples_leaf=8, l2_regularization=1.0,
             random_state=42)
XGB_KW = dict(n_estimators=600, learning_rate=0.04, tree_method="hist", grow_policy="lossguide", max_leaves=15,
              max_depth=0, min_child_weight=8, reg_lambda=1.0, random_state=42)


class MultiQuantile:
    # fits all quantile levels `taus`; predict(X) -> (n, len(taus)) sorted along axis 1 (no crossing)
    def __init__(self, taus, device=None):
        self.taus = list(taus); self.device = device or DEFAULT_DEVICE
        self.use_xgb = HAS_XGB and self.device.startswith("cuda")

    def _fit_sklearn(self, X, y):
        self.ms = [HistGradientBoostingRegressor(loss="quantile", quantile=t, **GB_KW).fit(X, y) for t in self.taus]
        self.use_xgb = False

    def fit(self, X, y):
        if self.use_xgb:
            try:
                self.xm = xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=np.array(self.taus),
                                           device=self.device, **XGB_KW).fit(X, y)
                return self
            except Exception as e:
                _warn_once("xgb", f"XGBoost GPU fit failed ({type(e).__name__}: {str(e)[:80]})")
        self._fit_sklearn(X, y)
        return self

    def predict(self, X):
        if self.use_xgb:
            P = np.asarray(self.xm.predict(X)).reshape(len(X), -1)
        else:
            P = np.column_stack([m.predict(X) for m in self.ms])
        return np.sort(P, axis=1)


# ---- Gaussian process: exact GP in torch on GPU, scikit-learn on CPU
def gp_interval(Xtr, ytr, Xte, z, ls0, device=None, iters=200, lr=0.05):
    device = device or DEFAULT_DEVICE
    if HAS_TORCH and device.startswith("cuda"):
        try:
            return _gp_torch(Xtr, ytr, Xte, z, ls0, device, iters, lr)
        except Exception as e:
            _warn_once("gp", f"torch GP failed ({type(e).__name__}: {str(e)[:80]})")
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as Ck
    gp = GaussianProcessRegressor(Ck(1.0) * RBF(list(ls0)) + WhiteKernel(0.1), normalize_y=True, random_state=42).fit(Xtr, ytr)
    m, s = gp.predict(Xte, return_std=True)
    return m - z * s, m + z * s


def _gp_torch(Xtr, ytr, Xte, z, ls0, device, iters, lr):
    dt = torch.float64
    X = torch.as_tensor(Xtr, dtype=dt, device=device); Xs = torch.as_tensor(Xte, dtype=dt, device=device)
    ym, ys = float(np.mean(ytr)), float(np.std(ytr) or 1.0)
    y = torch.as_tensor((np.asarray(ytr) - ym) / ys, dtype=dt, device=device)[:, None]
    log_ls = torch.tensor(np.log(ls0), dtype=dt, device=device, requires_grad=True)
    log_sf = torch.zeros((), dtype=dt, device=device, requires_grad=True)
    log_sn = torch.tensor(np.log(0.3), dtype=dt, device=device, requires_grad=True)
    eye = torch.eye(len(X), dtype=dt, device=device)

    def kern(A, B):
        ls = log_ls.exp()
        return torch.exp(2 * log_sf) * torch.exp(-0.5 * torch.cdist(A / ls, B / ls) ** 2)

    def chol(K):
        for jit in (1e-8, 1e-6, 1e-4):
            L, info = torch.linalg.cholesky_ex(K + jit * eye)
            if int(info) == 0:
                return L
        raise RuntimeError("Cholesky failed")

    opt = torch.optim.Adam([log_ls, log_sf, log_sn], lr=lr)
    for _ in range(iters):
        opt.zero_grad()
        L = chol(kern(X, X) + torch.exp(2 * log_sn) * eye)
        a = torch.cholesky_solve(y, L)
        nll = 0.5 * (y * a).sum() + torch.log(torch.diagonal(L)).sum()
        nll.backward(); opt.step()
    with torch.no_grad():
        L = chol(kern(X, X) + torch.exp(2 * log_sn) * eye)
        a = torch.cholesky_solve(y, L); Ks = kern(Xs, X)
        mu = (Ks @ a)[:, 0]
        v = torch.linalg.solve_triangular(L, Ks.T, upper=False)
        var = torch.exp(2 * log_sf) - (v ** 2).sum(0) + torch.exp(2 * log_sn)
        sd = torch.sqrt(var.clamp_min(1e-12))
    mu = mu.cpu().numpy() * ys + ym; sd = sd.cpu().numpy() * ys
    return mu - z * sd, mu + z * sd


# ---- MLP ensemble: torch on GPU, scikit-learn on CPU
def mlp_ensemble_interval(Xtr, ytr, Xte, z, n_models=5, device=None, seed=42):
    device = device or DEFAULT_DEVICE
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler().fit(Xtr); A, B = sc.transform(Xtr), sc.transform(Xte)
    ptr = pte = None
    if HAS_TORCH and device.startswith("cuda"):
        try:
            ptr, pte = _mlp_torch(A, np.asarray(ytr), B, n_models, device, seed)
        except Exception as e:
            _warn_once("mlp", f"torch MLP failed ({type(e).__name__}: {str(e)[:80]})")
    if ptr is None:
        from sklearn.neural_network import MLPRegressor
        ptr, pte = [], []
        for r in range(n_models):
            m = MLPRegressor(hidden_layer_sizes=(64, 64), alpha=1e-3, max_iter=3000, early_stopping=True,
                             random_state=seed + r).fit(A, ytr)
            ptr.append(m.predict(A)); pte.append(m.predict(B))
        ptr, pte = np.array(ptr), np.array(pte)
    sig2 = np.mean((np.asarray(ytr) - ptr.mean(0)) ** 2)
    s = np.sqrt(pte.var(0) + sig2); m = pte.mean(0)
    return m - z * s, m + z * s


def _mlp_torch(A, y, B, n_models, device, seed, max_epochs=3000, patience=100):
    dt = torch.float32
    Xa = torch.as_tensor(A, dtype=dt, device=device); Xb = torch.as_tensor(B, dtype=dt, device=device)
    ym, ys = y.mean(), y.std() or 1.0
    yt = torch.as_tensor((y - ym) / ys, dtype=dt, device=device)[:, None]
    ptr, pte = [], []
    for r in range(n_models):
        torch.manual_seed(seed + r)
        perm = np.random.default_rng(seed + r).permutation(len(A)); nv = max(1, len(A) // 10)
        iv, it = torch.as_tensor(perm[:nv], device=device), torch.as_tensor(perm[nv:], device=device)
        net = torch.nn.Sequential(torch.nn.Linear(A.shape[1], 64), torch.nn.ReLU(), torch.nn.Linear(64, 64),
                                  torch.nn.ReLU(), torch.nn.Linear(64, 1)).to(device)
        opt = torch.optim.Adam(net.parameters(), lr=3e-3, weight_decay=1e-4)
        best, best_state, wait = np.inf, None, 0
        for ep in range(max_epochs):
            opt.zero_grad()
            loss = torch.mean((net(Xa[it]) - yt[it]) ** 2); loss.backward(); opt.step()
            with torch.no_grad():
                v = float(torch.mean((net(Xa[iv]) - yt[iv]) ** 2))
            if v < best - 1e-6:
                best, wait = v, 0; best_state = {k: t.clone() for k, t in net.state_dict().items()}
            else:
                wait += 1
                if wait > patience:
                    break
        net.load_state_dict(best_state)
        with torch.no_grad():
            ptr.append(net(Xa)[:, 0].cpu().numpy() * ys + ym); pte.append(net(Xb)[:, 0].cpu().numpy() * ys + ym)
    return np.array(ptr), np.array(pte)


print(f"backend: {N_CPU} CPU core(s), {N_WORKERS} worker(s) | GPUs: {N_GPUS} | GPU on: {GPU_ON} "
      f"| torch: {HAS_TORCH} | xgboost>=2: {HAS_XGB} | default device: {DEFAULT_DEVICE}")

import multiprocessing

def inner_jobs():
    return -1 if multiprocessing.parent_process() is None else 1

In [ ]:
# ---- backend self-check (fast): GPU and CPU learners should give similar quantile coverage
if GPU_ON:
    rs = np.random.default_rng(0); Xc = rs.normal(size=(1500, 5)); yc = Xc[:, 0] + np.sin(2 * Xc[:, 1]) + 0.3 * rs.normal(size=1500)
    for dev in ["cpu", DEFAULT_DEVICE]:
        t0 = time.time(); P = MultiQuantile((0.05, 0.5, 0.95), device=dev).fit(Xc[:1000], yc[:1000]).predict(Xc[1000:])
        cov = np.mean((yc[1000:] >= P[:, 0]) & (yc[1000:] <= P[:, 2]))
        print(f"quantile GB on {dev:7s}: {time.time() - t0:5.2f}s, 90% band coverage {cov:.3f}")
    for dev in ["cpu", DEFAULT_DEVICE]:
        t0 = time.time(); lo, hi = gp_interval(Xc[:1000], yc[:1000], Xc[1000:], 1.645, [1.0] * 5, device=dev)
        print(f"GP on {dev:7s}: {time.time() - t0:5.2f}s, 90% coverage {np.mean((yc[1000:] >= lo) & (yc[1000:] <= hi)):.3f}")
else:
    print("no GPU: everything runs on CPU workers")

## 1. Data

In [ ]:
RAW = "https://raw.githubusercontent.com/yamenetoo/nuclear-mass-peel-cqr/main/data/ame/"
AME = {2003: ("mass.mas03", ["https://www-nds.iaea.org/amdc/ame2003/mass.mas03"]),
       2012: ("mass.mas12", [RAW + "mass.mas12", "https://www-nds.iaea.org/amdc/ame2012/mass.mas12"]),
       2016: ("mass16.txt", ["https://www-nds.iaea.org/amdc/ame2016/mass16.txt"]),
       2020: ("mass_1.mas20.txt", [RAW + "mass_1.mas20.txt", "https://www-nds.iaea.org/amdc/ame2020/mass_1.mas20.txt"])}
KEYS = {2003: ("mas03", "mass03"), 2012: ("mas12", "mass12"), 2016: ("mass16", "mas16"), 2020: ("mas20", "mass20")}


def get_ame(y):
    fname, urls = AME[y]; path = os.path.join(DATA_DIR, fname)
    if os.path.exists(path) and os.path.getsize(path) > 100_000:
        return path
    for root in ["data/ame", "../data/ame", "/kaggle/input", ".", "/content"]:
        if not os.path.isdir(root):
            continue
        for h in glob.glob(os.path.join(root, "**", "*"), recursive=True):
            n = os.path.basename(h).lower()
            if os.path.isfile(h) and any(k in n for k in KEYS[y]) and os.path.getsize(h) > 100_000:
                shutil.copy(h, path); return path
    for url in urls:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=20) as r:
                data = r.read()
            if len(data) > 100_000:
                open(path, "wb").write(data); return path
        except Exception as e:
            print(f"AME{y}: {url} -> {type(e).__name__}")
    return None


paths = {y: get_ame(y) for y in AME}
EDITIONS = [y for y in AME if paths[y]]
assert 2012 in EDITIONS and 2020 in EDITIONS, "AME2012 and AME2020 are required"
print("editions:", EDITIONS)

DELTA_H, DELTA_N = 7288.971064, 8071.318062


def parse_ame(path):
    rows = []
    for line in open(path, encoding="latin-1"):
        if len(line) < 40:
            continue
        try:
            N = int(line[4:9]); Z = int(line[9:14]); A = int(line[14:19])
        except ValueError:
            continue
        if N + Z != A:
            continue
        t = line[28:].split()
        if not t:
            continue
        try:
            me = float(t[0].replace("#", ""))
        except ValueError:
            continue
        rows.append((Z, N, A, me, "#" in t[0]))
    d = pd.DataFrame(rows, columns=["Z", "N", "A", "ME", "est"]).drop_duplicates(["Z", "N"])
    d["B"] = d.Z * DELTA_H + d.N * DELTA_N - d.ME
    return d.reset_index(drop=True)


ame = {y: parse_ame(paths[y]) for y in EDITIONS}
MEAS = {y: set(zip(*ame[y][(~ame[y].est) & (ame[y].Z >= MIN_Z) & (ame[y].N >= MIN_N)][["Z", "N"]].values.T)) for y in EDITIONS}
d20 = ame[2020]
data = d20[(d20.Z >= MIN_Z) & (d20.N >= MIN_N)].reset_index(drop=True)
data["key"] = list(zip(data.Z, data.N))
data["m20"] = data.key.map(lambda k: k in MEAS[2020])
ZN = data[["Z", "N"]].values
for y in EDITIONS:
    print(f"AME{y}: measured (Z,N>=8) {len(MEAS[y])}")

## 2. Base model, features, distances

In [ ]:
BASE_TABLE = None   # pd.DataFrame with Z, N, B_theory (keV) for WS4 / FRDM2012 / HFB


def ldm_design(Z, N):
    Z = np.asarray(Z, float); N = np.asarray(N, float); A = Z + N
    dl = np.where((Z % 2 == 0) & (N % 2 == 0), 1.0, np.where((Z % 2 == 1) & (N % 2 == 1), -1.0, 0.0))
    return np.column_stack([A, -A ** (2 / 3), -Z * (Z - 1) / A ** (1 / 3), -(N - Z) ** 2 / A,
                            (N - Z) ** 2 / A ** (4 / 3), dl * A ** -0.5])


if BASE_TABLE is not None:
    data = data.merge(BASE_TABLE, on=["Z", "N"], how="inner").reset_index(drop=True); ZN = data[["Z", "N"]].values


def base_prediction(train_mask):
    if BASE_TABLE is not None:
        return data.B_theory.values
    coef, *_ = np.linalg.lstsq(ldm_design(data.Z[train_mask], data.N[train_mask]), data.B[train_mask], rcond=None)
    return ldm_design(data.Z, data.N) @ coef


MAGIC = np.array([2, 8, 20, 28, 50, 82, 126, 184])


def valence(x):
    x = np.asarray(x)[:, None]
    lo = np.where(MAGIC[None] <= x, MAGIC[None], -1).max(1); hi = np.where(MAGIC[None] > x, MAGIC[None], 10**6).min(1)
    xv = x[:, 0]
    return np.minimum(xv - lo, hi - xv), (xv - lo) / (hi - lo)


def features(Z, N):
    Z = np.asarray(Z); N = np.asarray(N); A = Z + N
    vz, fz = valence(Z); vn, fn = valence(N)
    P = np.where(vz + vn > 0, vz * vn / np.maximum(vz + vn, 1), 0.0)
    return np.column_stack([Z, N, A, (N - Z) / A, Z % 2, N % 2, vz, vn, fz, fn, P]).astype(float)


X_ALL = features(data.Z.values, data.N.values)


def dist_to(ref, pts, metric=None):
    metric = metric or DIST_METRIC
    ref = np.asarray(ref); pts = np.asarray(pts); out = np.empty(len(pts))
    for i in range(0, len(pts), 512):
        dz = np.abs(pts[i:i + 512, None, 0] - ref[None, :, 0]); dn = np.abs(pts[i:i + 512, None, 1] - ref[None, :, 1])
        if metric == "manhattan":
            dd = dz + dn
        elif metric == "euclidean":
            dd = np.sqrt(dz ** 2 + dn ** 2)
        else:
            dd = np.maximum(dz, dn)
        out[i:i + 512] = dd.min(1)
    return np.ceil(out - 1e-9).astype(int)


def depth_in(zn, metric=None, min_zn=None):
    mz, mn = min_zn or (MIN_Z, MIN_N)
    zn = np.asarray(zn); zmax, nmax = zn[:, 0].max() + 8, zn[:, 1].max() + 8
    g = np.zeros((zmax + 1, nmax + 1), bool); g[zn[:, 0], zn[:, 1]] = True
    holes = np.argwhere(~g); holes = holes[(holes[:, 0] >= mz - 1) & (holes[:, 1] >= mn - 1)]
    return dist_to(holes, zn, metric)

## 3. Learners and interval methods

Every method returns `(lo, hi)` for the test points. Weighted conformal prediction returns an infinite interval when the test point's weight dominates all calibration weights — which happens when test inputs lie outside the support of the training inputs, exactly the extrapolation setting. Summaries report the fraction of infinite intervals and compute widths over finite ones. Quantile learners expose `fit(X, y)` and `predict3(X) -> (lo, med, hi)`.

| Method | Type | Uses later edition | Distance-aware |
|---|---|---|---|
| GP | Bayesian | no | implicitly |
| MLP-ensemble | ensemble of 5 MLPs, Gaussian interval | no | implicitly |
| QRF | quantile regression forest (raw) | no | no |
| CV+ | cross-conformal, gradient boosting mean model | no | no |
| CQR-random | split conformal, random calibration | no | no |
| WCP | weighted CQR, density ratio from a classifier (exact; intervals can be infinite) | no (uses test inputs) | via weights |
| WCP (clip) | as WCP with classifier probabilities clipped to [0.01, 0.99] | no | via weights |
| CQR-temporal | split conformal, next edition | yes | no |
| Peel-CQR (GB) | ours, gradient-boosted quantiles | no | yes |
| Peel-CQR (QRF) | ours, quantile regression forest | no | yes |

In [ ]:
GB_KW = dict(max_iter=600, learning_rate=0.04, max_leaf_nodes=15, min_samples_leaf=8, l2_regularization=1.0,
             random_state=SEED)
Z90 = stats.norm.ppf(1 - ALPHA / 2)


class QuantileGB:
    # gradient-boosted quantiles at alpha/2, 0.5, 1-alpha/2 (XGBoost on GPU, scikit-learn on CPU)
    name = "GB"

    def __init__(self, alpha=ALPHA, device=None):
        self.m = MultiQuantile((alpha / 2, 0.5, 1 - alpha / 2), device)

    def fit(self, X, y):
        self.m.fit(X, y); return self

    def predict3(self, X):
        P = self.m.predict(X)
        return P[:, 0], P[:, 1], P[:, 2]


class QRF:
    # quantile regression forest (Meinshausen 2006): weighted empirical quantiles of training responses
    name = "QRF"

    def __init__(self, alpha=ALPHA, n_estimators=200, min_samples_leaf=5, device=None):
        self.taus = (alpha / 2, 0.5, 1 - alpha / 2); self.device = device or DEFAULT_DEVICE
        self.rf = RandomForestRegressor(n_estimators=n_estimators, min_samples_leaf=min_samples_leaf,
                                        max_features=0.6, random_state=SEED, n_jobs=inner_jobs())

    def fit(self, X, y):
        self.rf.fit(X, y); self.y = np.asarray(y); self.leaves = self.rf.apply(X)
        self.order = np.argsort(self.y); self.ys = self.y[self.order]
        return self

    def _weights_torch(self, L):
        dev = self.device
        Lt = torch.as_tensor(L, device=dev); Ft = torch.as_tensor(self.leaves, device=dev)
        W = torch.zeros((len(L), len(self.y)), dtype=torch.float32, device=dev)
        for t in range(L.shape[1]):
            match = (Lt[:, t, None] == Ft[None, :, t]).float()
            W += match / match.sum(1, keepdim=True).clamp_min(1)
        return (W / L.shape[1]).cpu().numpy()

    def predict3(self, X, taus=None):
        taus = taus or self.taus
        L = self.rf.apply(X); n, T = len(self.y), L.shape[1]
        out = np.zeros((len(X), len(taus)))
        if HAS_TORCH and self.device.startswith("cuda"):
            try:
                cw = np.cumsum(self._weights_torch(L)[:, self.order], axis=1)
                for j, tau in enumerate(taus):
                    out[:, j] = self.ys[np.minimum((cw < tau).sum(1), n - 1)]
                out = np.sort(out, axis=1)
                return out[:, 0], out[:, 1], out[:, 2]
            except Exception as e:
                _warn_once("qrf", f"torch QRF weights failed ({type(e).__name__})")
        for s in range(0, len(X), 400):
            W = np.zeros((min(400, len(X) - s), n))
            for t in range(T):
                match = (L[s:s + 400, t][:, None] == self.leaves[None, :, t])
                W += match / np.maximum(match.sum(1, keepdims=True), 1)
            W /= T
            cw = np.cumsum(W[:, self.order], axis=1)
            for j, tau in enumerate(taus):
                idx = np.minimum((cw < tau).sum(1), n - 1)
                out[s:s + 400, j] = self.ys[idx]
        out = np.sort(out, axis=1)
        return out[:, 0], out[:, 1], out[:, 2]


def cqr_scores(model, X, y):
    lo, _, hi = model.predict3(X)
    return np.maximum(lo - y, y - hi)


def conformal_q(s, alpha=ALPHA):
    s = np.sort(np.asarray(s)); n = len(s)
    if n == 0:
        return np.nan
    k = int(np.ceil((n + 1) * (1 - alpha)))
    return s[min(k, n) - 1]


def weighted_q(scores, w_cal, w_test, alpha=ALPHA):
    # weighted conformal quantile (Tibshirani et al. 2019) for each test point
    o = np.argsort(scores); s = scores[o]; wc = w_cal[o]
    out = np.empty(len(w_test))
    csum = np.cumsum(wc)
    for i, wt in enumerate(w_test):
        p = csum / (csum[-1] + wt)
        j = np.searchsorted(p, 1 - alpha)
        out[i] = s[j] if j < len(s) else np.inf
    return out


def peel_calibrate(learner_cls, Xtr, ytr, zn_tr, domain_zn=None, K=None, metric=None, monotone=True, min_zn=None):
    # learner_cls: a zero-argument factory, e.g. functools.partial(QuantileGB, device=device)
    K = K or K_PEEL
    dom = zn_tr if domain_zn is None else domain_zn
    dmap = dict(zip(map(tuple, dom), depth_in(dom, metric, min_zn)))
    dep = np.array([dmap.get(tuple(p), 1) for p in zn_tr]); core = dep > K
    if core.sum() < 50:
        raise ValueError(f"core too small ({core.sum()} nuclei) for K={K}, metric={metric or DIST_METRIC}")
    qc = learner_cls().fit(Xtr[core], ytr[core])
    dcore = dist_to(zn_tr[core], zn_tr[~core], metric)
    s = cqr_scores(qc, Xtr[~core], ytr[~core])
    ks = np.arange(1, K + 1)
    nd = np.array([(dcore == k).sum() for k in ks])
    cd = np.array([conformal_q(s[dcore == k]) if (dcore == k).sum() >= 20 else np.nan for k in ks])
    cd = pd.Series(cd).ffill().bfill().values
    if monotone:
        cd = np.maximum.accumulate(cd)
    ok = np.isfinite(cd)
    slope = max(0.0, np.polyfit(ks[ok], cd[ok], 1)[0]) if ok.sum() >= 2 else 0.0
    return dict(c=cd, slope=slope, n_d=nd, n_core=int(core.sum()))


def peel_c(p, d):
    d = np.maximum(np.asarray(d), 1); K = len(p["c"])
    return np.where(d <= K, p["c"][np.minimum(d, K) - 1], p["c"][-1] + p["slope"] * (d - K))

In [ ]:
def run_fold(e_tr, e_cal, e_te, methods=None, sub_idx=None, metric=None, monotone=True, K=None, verbose=True, device=None):
    device = device or DEFAULT_DEVICE
    QGB = functools.partial(QuantileGB, device=device); QRFd = functools.partial(QRF, device=device)
    times = {}
    keys = data.key.values; m20 = data.m20.values
    tr = np.array([k in MEAS[e_tr] for k in keys]) & m20
    domain = ZN[tr]
    if sub_idx is not None:
        idx = np.flatnonzero(tr); keep = np.zeros_like(tr); keep[idx[sub_idx]] = True; tr = keep
    prev = MEAS[e_cal] if e_cal else MEAS[e_tr]
    cal = (np.array([(k in MEAS[e_cal]) and (k not in MEAS[e_tr]) for k in keys]) & m20) if e_cal else np.zeros(len(data), bool)
    te = np.array([(k in MEAS[e_te]) and (k not in prev) and (k not in MEAS[e_tr]) for k in keys]) & m20
    yv = (data.B.values - base_prediction(tr)) / 1000.0
    Xtr, ytr, zntr = X_ALL[tr], yv[tr], ZN[tr]
    Xte, yte = X_ALL[te], yv[te]
    dte = dist_to(zntr, ZN[te], metric)
    methods = methods or ["GP", "MLP-ensemble", "QRF", "CV+", "CQR-random", "WCP", "WCP (clip)", "CQR-temporal", "Peel-CQR (GB)", "Peel-CQR (QRF)"]
    out, info = {}, {"fold": f"{e_tr}->{e_cal or '-'}->{e_te}", "n_train": int(tr.sum()), "n_cal": int(cal.sum()), "n_test": int(te.sum())}
    i_fit, i_cal = train_test_split(np.arange(len(ytr)), test_size=0.2, random_state=SEED)

    def timed(name, fn):
        t0 = time.time(); res = fn(); times.setdefault(name, []).append(time.time() - t0); return res

    if "GP" in methods and RUN_GP:
        def f():
            cols = [0, 1, 4, 5]; mu, sd = Xtr[:, cols].mean(0), Xtr[:, cols].std(0)
            return gp_interval((Xtr[:, cols] - mu) / sd, ytr, (Xte[:, cols] - mu) / sd, Z90, [0.2, 0.2, 1.0, 1.0], device)
        out["GP"] = timed("GP", f)
    if "MLP-ensemble" in methods:
        out["MLP-ensemble"] = timed("MLP-ensemble", lambda: mlp_ensemble_interval(Xtr, ytr, Xte, Z90, device=device, seed=SEED))
    if "QRF" in methods or "Peel-CQR (QRF)" in methods:
        qrf_full = timed("QRF (fit)", lambda: QRFd().fit(Xtr, ytr))
        if "QRF" in methods:
            lo, _, hi = qrf_full.predict3(Xte); out["QRF"] = (lo, hi)
    if "CV+" in methods:
        def f():
            kf = KFold(10, shuffle=True, random_state=SEED); mu_te = np.zeros((len(ytr), len(yte))); R = np.zeros(len(ytr))
            for a, b in kf.split(Xtr):
                m = HistGradientBoostingRegressor(**GB_KW).fit(Xtr[a], ytr[a])
                R[b] = np.abs(ytr[b] - m.predict(Xtr[b])); mu_te[b] = m.predict(Xte)[None, :]
            lo = np.quantile(mu_te - R[:, None], ALPHA, axis=0, method="lower")
            hi = np.quantile(mu_te + R[:, None], 1 - ALPHA, axis=0, method="higher")
            return lo, hi
        out["CV+"] = timed("CV+", f)
    if "CQR-random" in methods or "WCP" in methods or "WCP (clip)" in methods:
        qr = timed("CQR-random", lambda: QGB().fit(Xtr[i_fit], ytr[i_fit]))
        s_r = cqr_scores(qr, Xtr[i_cal], ytr[i_cal]); lr, _, hr = qr.predict3(Xte)
        if "CQR-random" in methods:
            c = conformal_q(s_r); out["CQR-random"] = (lr - c, hr + c)
        if "WCP" in methods or "WCP (clip)" in methods:
            Xc = np.vstack([Xtr, Xte]); lab = np.r_[np.zeros(len(Xtr)), np.ones(len(Xte))]
            clf = timed("WCP", lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                                                      random_state=SEED).fit(Xc, lab))
            for name, eps in [("WCP", 1e-6), ("WCP (clip)", 0.01)]:
                if name in methods:
                    pc = np.clip(clf.predict_proba(Xtr[i_cal])[:, 1], eps, 1 - eps)
                    pt = np.clip(clf.predict_proba(Xte)[:, 1], eps, 1 - eps)
                    cw = weighted_q(s_r, pc / (1 - pc), pt / (1 - pt))     # may be +inf: no finite interval
                    out[name] = (lr - cw, hr + cw)
    qgb = timed("GB quantiles (fit)", lambda: QGB().fit(Xtr, ytr))
    lo, med, hi = qgb.predict3(Xte)
    if "CQR-temporal" in methods and cal.any():
        c = conformal_q(cqr_scores(qgb, X_ALL[cal], yv[cal])); out["CQR-temporal"] = (lo - c, hi + c)
    if "Peel-CQR (GB)" in methods:
        p = timed("Peel-CQR (GB)", lambda: peel_calibrate(QGB, Xtr, ytr, zntr, domain, K, metric, monotone))
        c = peel_c(p, dte); out["Peel-CQR (GB)"] = (lo - c, hi + c)
        info.update({f"peelGB_c{k+1}": v for k, v in enumerate(p["c"])}); info["peelGB_slope"] = p["slope"]
    if "Peel-CQR (QRF)" in methods:
        p2 = timed("Peel-CQR (QRF)", lambda: peel_calibrate(QRFd, Xtr, ytr, zntr, domain, K, metric, monotone))
        lq, _, hq = qrf_full.predict3(Xte); c = peel_c(p2, dte); out["Peel-CQR (QRF)"] = (lq - c, hq + c)
    pred = pd.DataFrame({"fold": info["fold"], "Z": ZN[te, 0], "N": ZN[te, 1], "d": dte, "y": yte, "median": med})
    for k, (a, b) in out.items():
        pred[f"{k}|lo"] = a; pred[f"{k}|hi"] = b
    if verbose:
        print(info)
    return pred, info, times


def fold_task(fold, device=None, **kw):
    return run_fold(*fold, device=device, verbose=False, **kw)

## 4. Rolling folds on real data

In [ ]:
FOLDS = []
if {2003, 2012, 2016} <= set(EDITIONS): FOLDS.append((2003, 2012, 2016))
if {2012, 2016, 2020} <= set(EDITIONS): FOLDS.append((2012, 2016, 2020))
if {2003, 2012} <= set(EDITIONS): FOLDS.append((2003, None, 2012))
if 2016 not in EDITIONS: FOLDS.append((2012, None, 2020))
print("folds:", FOLDS)
t0 = time.time()
res = pmap(fold_task, [dict(fold=f) for f in FOLDS], desc="rolling folds")
TIMES = {}
for _, _, tm in res:
    for k, v in tm.items():
        TIMES.setdefault(k, []).extend(v)
pred_all = pd.concat([r[0] for r in res], ignore_index=True); fold_info = pd.DataFrame([r[1] for r in res])
for i in fold_info.to_dict("records"):
    print(i)
METHODS = [c.split("|")[0] for c in pred_all.columns if c.endswith("|lo")]
print(f"{time.time()-t0:.0f}s, pooled test nuclei {len(pred_all)}, methods {METHODS}")

In [ ]:
def cp(k, n, a=0.05):
    if n == 0:
        return np.nan, np.nan
    return (0.0 if k == 0 else stats.beta.ppf(a / 2, k, n - k + 1), 1.0 if k == n else stats.beta.ppf(1 - a / 2, k + 1, n - k))


def interval_score(y, lo, hi, alpha=ALPHA):
    return (hi - lo) + (2 / alpha) * ((lo - y) * (y < lo) + (y - hi) * (y > hi))


def summarize(df, by=None):
    rows = []
    groups = [("all", df)] if by is None else list(df.groupby(by))
    for g, sub in groups:
        for m in METHODS:
            lo, hi = sub[f"{m}|lo"], sub[f"{m}|hi"]; ok = lo.notna()
            if ok.sum() == 0:
                continue
            y = sub.y[ok]; lo, hi = lo[ok], hi[ok]
            cov = (y >= lo) & (y <= hi); k, n = int(cov.sum()), int(ok.sum()); a, b = cp(k, n)
            fin = np.isfinite(hi - lo)
            rows.append({(by or "group"): g, "method": m, "n": n, "coverage": k / n, "CP_lo": a, "CP_hi": b,
                         "p_vs_0.90": stats.binomtest(k, n, 1 - ALPHA).pvalue, "frac_infinite": float(1 - fin.mean()),
                         "mean_width": float(np.mean((hi - lo)[fin])) if fin.any() else np.inf,
                         "interval_score": float(np.mean(interval_score(y, lo, hi)[fin])) if fin.any() else np.inf})
    return pd.DataFrame(rows)


pooled = summarize(pred_all)
per_fold = summarize(pred_all, "fold")
pooled.sort_values("interval_score").round(3)

In [ ]:
per_fold.round(3)

## 5. Coverage by distance and conditional coverage by region

In [ ]:
pred_all["d_bin"] = np.where(pred_all.d >= 4, "4+", pred_all.d.astype(str))
by_d = summarize(pred_all, "d_bin")


def region(z, n):
    near_magic = (np.min(np.abs(MAGIC - z)) <= 2) or (np.min(np.abs(MAGIC - n)) <= 2)
    parity = "even-even" if (z % 2 == 0 and n % 2 == 0) else ("odd-odd" if (z % 2 and n % 2) else "odd-A")
    return pd.Series({"magic": "near magic" if near_magic else "mid-shell", "parity": parity,
                      "mass": "A<100" if z + n < 100 else ("100<=A<180" if z + n < 180 else "A>=180")})


pred_all = pd.concat([pred_all, pred_all.apply(lambda r: region(int(r.Z), int(r.N)), axis=1)], axis=1)
cond = pd.concat([summarize(pred_all, g).rename(columns={g: "region"}).assign(split=g) for g in ["magic", "parity", "mass"]])
fig, ax = plt.subplots(figsize=(9, 4))
order = sorted(set(by_d.d_bin))
for j, m in enumerate(METHODS):
    q = by_d[by_d.method == m].set_index("d_bin").reindex(order)
    x = np.arange(len(order)) + (j - len(METHODS) / 2) * 0.07
    ax.errorbar(x, q.coverage, yerr=[q.coverage - q.CP_lo, q.CP_hi - q.coverage], fmt="o", ms=3, capsize=2, label=m)
ax.axhline(1 - ALPHA, color="k", ls="--", lw=0.8); ax.set_xticks(range(len(order))); ax.set_xticklabels(order)
ax.set_xlabel("extrapolation distance d"); ax.set_ylabel("coverage"); ax.legend(fontsize=6, ncol=3); plt.tight_layout(); plt.show()
cond.pivot_table(index=["split", "region"], columns="method", values="coverage").round(3)

## 6. Paired tests with Holm correction

Exact McNemar test of Peel-CQR (GB) against every other method on the same test nuclei; p-values Holm-adjusted within each fold.

In [ ]:
rows = []
for fold, g in pred_all.groupby("fold"):
    ref = (g.y >= g["Peel-CQR (GB)|lo"]) & (g.y <= g["Peel-CQR (GB)|hi"])
    for m in METHODS:
        if m == "Peel-CQR (GB)" or g[f"{m}|lo"].isna().all():
            continue
        oth = (g.y >= g[f"{m}|lo"]) & (g.y <= g[f"{m}|hi"])
        b, c = int((ref & ~oth).sum()), int((~ref & oth).sum())
        p = stats.binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
        rows.append(dict(fold=fold, vs=m, b_peel_only=b, c_other_only=c, p=p,
                         width_ratio=(g["Peel-CQR (GB)|hi"] - g["Peel-CQR (GB)|lo"]).mean() / (g[f"{m}|hi"] - g[f"{m}|lo"]).replace(np.inf, np.nan).mean()))
mcn = pd.DataFrame(rows)


def holm(p):
    p = np.asarray(p); o = np.argsort(p); m = len(p); adj = np.empty(m)
    adj[o] = np.minimum(1, np.maximum.accumulate((m - np.arange(m)) * p[o]))
    return adj


mcn["p_holm"] = mcn.groupby("fold").p.transform(holm)
mcn.round(4)

## 7. Ablations on the most recent fold: distance metric, monotone constraint, K

In [ ]:
main_fold = FOLDS[1] if len(FOLDS) > 1 and FOLDS[1][2] == 2020 else [f for f in FOLDS if f[2] == 2020][0]
def ablation_task(metric, mono, K, device=None):
    try:
        p, i, _ = run_fold(*main_fold, methods=["Peel-CQR (GB)"], metric=metric, monotone=mono, K=K, verbose=False, device=device)
    except ValueError as e:
        return dict(metric=metric, monotone=mono, K=K, coverage=np.nan, width=np.nan, interval_score=np.nan, note=str(e))
    return dict(metric=metric, monotone=mono, K=K,
                coverage=((p.y >= p["Peel-CQR (GB)|lo"]) & (p.y <= p["Peel-CQR (GB)|hi"])).mean(),
                width=(p["Peel-CQR (GB)|hi"] - p["Peel-CQR (GB)|lo"]).mean(),
                interval_score=np.mean(interval_score(p.y, p["Peel-CQR (GB)|lo"], p["Peel-CQR (GB)|hi"])))


abl = pmap(ablation_task, [dict(metric=m, mono=mo, K=K) for m in ["manhattan", "euclidean", "chebyshev"]
                           for mo in [True, False] for K in [2, 3, 4]], desc="ablations")
ablation = pd.DataFrame(abl)
ablation.round(3)

In [ ]:
timing = pd.DataFrame([(k, np.mean(v), len(v)) for k, v in TIMES.items()], columns=["step", "mean_seconds", "calls"]).sort_values("mean_seconds")
timing.round(2)

## 8. Simulation study with known truth

A synthetic chart: for each $Z\in[10,90]$ a stability line $N_s(Z)=1.1Z+0.004Z^2$ and offset $u=(N-N_s)/10$. Truth
$$f(Z,N)=2\sin(Z/6)\cos(N/8) + 0.8\,(\text{-1})^{Z+N} + \kappa\,u^2\sin(N/3),$$
noise $\varepsilon\sim\mathcal N(0,\,[0.25(1+|u|)]^2)$. "Editions" are nested bands $|N-N_s|\le 6,\,7,\,8$ with 5% random interior gaps (measured "later"; the peeling frontier is the edge of the band). Larger κ makes the function rougher away from stability, i.e. harder extrapolation. Each replication draws new noise and gaps.

In [ ]:
def simulate(kappa, rep):
    r = np.random.default_rng(1000 + rep)
    rows = []
    for Z in range(10, 91):
        Ns = 1.1 * Z + 0.004 * Z ** 2
        for N in range(int(Ns) - 9, int(Ns) + 10):
            u = (N - Ns) / 10
            f = 2 * np.sin(Z / 6) * np.cos(N / 8) + 0.8 * (-1) ** (Z + N) + kappa * u ** 2 * np.sin(N / 3)
            rows.append((Z, N, abs(N - Ns), f + r.normal(0, 0.25 * (1 + abs(u)))))
    d = pd.DataFrame(rows, columns=["Z", "N", "off", "y"])
    gap = r.random(len(d)) < 0.05
    d["ed"] = np.select([(d.off <= 6) & ~gap, (d.off <= 7), (d.off <= 8)], [0, 1, 2], default=3)
    return d


def sim_run(kappa, rep, device=None):
    QGB = functools.partial(QuantileGB, device=device)
    d = simulate(kappa, rep)
    X = np.column_stack([d.Z, d.N, d.Z % 2, d.N % 2, (d.N - d.Z) / (d.N + d.Z)]).astype(float)
    zn = d[["Z", "N"]].values; y = d.y.values
    tr, ca, te = (d.ed == 0).values, (d.ed == 1).values, (d.ed == 2).values
    q = QGB().fit(X[tr], y[tr]); lo, _, hi = q.predict3(X[te]); res = {}
    i_fit, i_cal = train_test_split(np.flatnonzero(tr), test_size=0.2, random_state=rep)
    qr = QGB().fit(X[i_fit], y[i_fit]); s_r = cqr_scores(qr, X[i_cal], y[i_cal]); lr, _, hr = qr.predict3(X[te])
    c = conformal_q(s_r); res["CQR-random"] = (lr - c, hr + c)
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=rep).fit(
        np.vstack([X[tr], X[te]]), np.r_[np.zeros(tr.sum()), np.ones(te.sum())])
    for name, eps in [("WCP", 1e-6), ("WCP (clip)", 0.01)]:
        pc = np.clip(clf.predict_proba(X[i_cal])[:, 1], eps, 1 - eps); pt = np.clip(clf.predict_proba(X[te])[:, 1], eps, 1 - eps)
        cw = weighted_q(s_r, pc / (1 - pc), pt / (1 - pt)); res[name] = (lr - cw, hr + cw)
    c = conformal_q(cqr_scores(q, X[ca], y[ca])); res["CQR-temporal"] = (lo - c, hi + c)
    footprint = zn[(d.off <= 6).values]          # frontier = edge of the band; interior gaps are not frontier
    p = peel_calibrate(QGB, X[tr], y[tr], zn[tr], domain_zn=footprint, min_zn=(10, int(zn[:, 1].min())))
    c = peel_c(p, dist_to(zn[tr], zn[te])); res["Peel-CQR (GB)"] = (lo - c, hi + c)
    out = []
    for m, (a, b) in res.items():
        fin = np.isfinite(b - a)
        out.append(dict(kappa=kappa, rep=rep, method=m, coverage=np.mean((y[te] >= a) & (y[te] <= b)),
                        frac_infinite=1 - fin.mean(), width=np.mean((b - a)[fin]) if fin.any() else np.inf,
                        interval_score=np.mean(interval_score(y[te], a, b)[fin]) if fin.any() else np.inf))
    return out


sim = pd.DataFrame()
if RUN_SIMULATION:
    runs = pmap(sim_run, [dict(kappa=k, rep=r) for k in SIM_KAPPAS for r in range(N_SIM_REPS)], desc="simulation")
    sim = pd.DataFrame([row for run in runs for row in run])
    sim_summary = sim.groupby(["kappa", "method"])[["coverage", "frac_infinite", "width", "interval_score"]].agg(["mean", "std"])
else:
    sim_summary = pd.DataFrame()
sim_summary.round(3)

In [ ]:
if len(sim):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for m, g in sim.groupby("method"):
        s = g.groupby("kappa").coverage.agg(["mean", "std"]); w = g.replace(np.inf, np.nan).groupby("kappa").width.mean()
        axes[0].errorbar(s.index, s["mean"], yerr=s["std"], marker="o", capsize=3, label=m)
        axes[1].plot(w.index, w.values, marker="o", label=m)
    axes[0].axhline(1 - ALPHA, color="k", ls="--", lw=0.8); axes[0].set_xlabel("extrapolation difficulty κ"); axes[0].set_ylabel("coverage (mean ± sd)")
    axes[1].set_xlabel("κ"); axes[1].set_ylabel("mean width"); axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()

## 9. Application: where should the next masses be measured?

Fit on all nuclei measured in AME2020, calibrate by peeling, and rank the unmeasured nuclei inside the calibrated range ($d\le K$) by the width of their 90% Peel-CQR interval. Wide calibrated intervals mark the measurements that would reduce uncertainty most.

In [ ]:
allm = data.m20.values
yall = (data.B.values - base_prediction(allm)) / 1000.0
q_all = QuantileGB().fit(X_ALL[allm], yall[allm])
p_all = peel_calibrate(QuantileGB, X_ALL[allm], yall[allm], ZN[allm])     # default device
cand = data[~allm].copy()
cand["d"] = dist_to(ZN[allm], cand[["Z", "N"]].values)
cand = cand[cand.d <= K_PEEL]
lo, med, hi = q_all.predict3(features(cand.Z.values, cand.N.values))
cand["width90_MeV"] = (hi - lo) + 2 * peel_c(p_all, cand.d.values)
SYM = ("n H He Li Be B C N O F Ne Na Mg Al Si P S Cl Ar K Ca Sc Ti V Cr Mn Fe Co Ni Cu Zn Ga Ge As Se Br Kr Rb Sr Y Zr "
       "Nb Mo Tc Ru Rh Pd Ag Cd In Sn Sb Te I Xe Cs Ba La Ce Pr Nd Pm Sm Eu Gd Tb Dy Ho Er Tm Yb Lu Hf Ta W Re Os Ir Pt "
       "Au Hg Tl Pb Bi Po At Rn Fr Ra Ac Th Pa U Np Pu Am Cm Bk Cf Es Fm Md No Lr Rf Db Sg Bh Hs Mt Ds Rg Cn Nh Fl Mc Lv Ts Og").split()
cand["nuclide"] = [f"{a}{SYM[z] if z < len(SYM) else z}" for z, a in zip(cand.Z, cand.A)]
priority = cand.sort_values("width90_MeV", ascending=False)[["nuclide", "Z", "N", "A", "d", "width90_MeV"]].head(25).reset_index(drop=True)
print(f"peeling on AME2020: c_d = {np.round(p_all['c'], 3)}, slope {p_all['slope']:.3f}; candidates within d<={K_PEEL}: {len(cand)}")
priority.round(3)

## 10. Save all results

In [ ]:
XLSX = "nuclear_mass_results_v3.xlsx"
backend_info = pd.DataFrame({"setting": ["CPU cores", "workers", "GPUs", "GPU used", "torch", "xgboost>=2"],
                             "value": [N_CPU, N_WORKERS, N_GPUS, GPU_ON, HAS_TORCH, HAS_XGB]})
sheets = {"backend": backend_info, "pooled": pooled, "per_fold": per_fold, "by_distance": by_d, "conditional": cond, "mcnemar_holm": mcn,
          "ablation": ablation, "timing": timing, "fold_info": fold_info,
          "simulation_runs": sim, "simulation_summary": sim_summary.reset_index() if len(sim_summary) else sim_summary,
          "priority_measurements": priority, "test_predictions": pred_all}
with pd.ExcelWriter(XLSX, engine="openpyxl") as xw:
    for name, df in sheets.items():
        if isinstance(df.columns, pd.MultiIndex):
            df = df.copy(); df.columns = [" ".join(map(str, c)).strip() for c in df.columns]
        df.round(4).to_excel(xw, sheet_name=name[:31], index=False)
    for ws in xw.book.worksheets:
        ws.freeze_panes = "A2"
print("saved", XLSX)